# Week 04 ? Task 05: Written justification

## Leadership review figure

**Panel 1 ? daily sessions.** The question is whether traffic has a trend after the weekday cycle is removed. A line chart preserves temporal order and makes continuity visible; the faint raw series prevents smoothing from hiding noise. It follows the Week 2 channel rule for ordered time: position carries sequence, with one shared quantitative axis. The y-axis says **Sessions per day (sum)**.

**Panel 2 ? weekday ? hour heatmap.** The question is when traffic concentrates across the week. A heatmap is appropriate because two categorical time dimensions form a compact grid and colour encodes a single quantitative measure. It follows the channel rule that colour is used for magnitude on a common sequential scale; the legend says **Mean sessions per recorded hour**.

**Panel 3 ? category/sub-category treemap.** The question is where the net-revenue mass is, including the hierarchy from parent category to sub-category. A treemap uses area to show part-to-whole mass and adjacency/containment to show hierarchy; it follows the Week 2 area rule by using only non-negative values and a declared whole. One hue family is used per parent category.

**Panel 4 ? sorted horizontal bars.** The question is by how much each sub-category differs. Bars are better than area for precise ranking and comparison because a common baseline makes lengths readable; this follows the Week 2 position/length rule. Values are printed and the x-axis says **Net revenue (PKR m; sum)**.

## Time decisions

The timestamp column was parsed with `format="mixed", dayfirst=True`. The file mixes ISO timestamps such as `2024-04-29 11:00:00` with slash dates such as `05/10/2024 09:00`; `dayfirst=True` resolves the slash ambiguity as day/month/year while leaving ISO year-month-day values correctly ordered. This was checked against the expected range: 4 January 2024 to 8 December 2025. A successful parse alone was not accepted because mixed parsing without the day-first decision silently shifts ambiguous dates.

The resolution is daily. After duplicate removal, hourly sessions are resampled with **sum** because sessions are a flow/count accumulated across hours; a level variable would not be summed. Missing hourly periods were counted before resampling, and the daily index was reindexed to a complete calendar. The final December 2025 month has only eight observed days, so it was excluded rather than presented as a complete month.

The smoothing window was fixed at **7 days before plotting**, because the cycle being removed is weekly (Part B.1). The series begins at its first recorded day; no start date was selected after inspecting the trend. The review figure uses a **7-day centred rolling mean** because this is a retrospective Monday leadership review and the estimate may use the complete seven-day neighbourhood; the raw daily series remains visible. A live dashboard would instead use a 7-day trailing mean and label it causal. I would have made the same 7-day choice if it weakened the conclusion: it is defined by the weekly cycle, not by which window looks better.

## Rows removed and rules

- **290 traffic rows:** `sessions < 0` were sentinels, removed rather than treated as real traffic. Traffic moved from **12,670 to 12,380 rows**.
- **886 traffic rows:** repeated timestamps were deduplicated, keeping the first record because duplicate records represented repeated copies of an hourly observation. Traffic moved from **12,380 to 11,494 hourly rows**.
- **Eight calendar days:** the partial final month, December 2025, was excluded from the daily review series because only 8 of 31 days were observed. This is a time-period exclusion, not silent deletion of source rows.
- **One catalogue row:** duplicate SKU `SKU-0127` was counted once. Catalogue rows moved from **345 to 344**.
- **Seven catalogue rows:** negative `net_revenue_pkr_m` values were excluded from treemap/bar area or length displays because negative values have no area and would distort a positive part-to-whole hierarchy. Catalogue rows moved from **344 to 337**.

Device labels were normalized by case (`Mobile`, `Desktop`, `Tablet`) rather than removing rows. No traffic rows were removed for missing timestamps: all timestamps parsed successfully.

## Roll-up depth and `Other`

The catalogue was rolled from SKU to **sub-category**, then grouped under its parent **category** for the treemap. The displayed hierarchy contains **8 sub-categories** under **3 parent categories**. I cut the display at sub-category depth because that is the level needed for the ranking bars and because deeper SKU labels would make the 2?2 review figure unreadable. There is **no `Other` bucket**: all eight sub-categories are shown, so zero categories and zero revenue are hidden. The positive cleaned net-revenue total is **PKR 1,712.4508m**.

The choice is symmetric: I did not choose the 7-day window, the first-date start, or the sub-category cut after seeing which version strengthened a claim. The complete raw/cleaned series, the named aggregation, the raw line, the removed-row counts, and the declared hierarchy make the decisions reproducible.

## Reproducibility check

The following cell verifies the key counts and the hierarchy rule against the source files.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path.cwd()
if not (DATA / "sessions_hourly.csv").exists():
    DATA = Path(r"D:\University\ML Lab\Week04")

traffic = pd.read_csv(DATA / "sessions_hourly.csv")
traffic["timestamp"] = pd.to_datetime(traffic["ts"], format="mixed", dayfirst=True)
nonnegative = traffic.loc[traffic.sessions >= 0]
unique_hours = nonnegative.drop_duplicates("timestamp")
assert len(traffic) == 12670
assert len(nonnegative) == 12380
assert len(unique_hours) == 11494

catalogue = pd.read_csv(DATA / "revenue_catalogue.csv")
catalogue = catalogue.drop_duplicates("sku", keep="first")
positive = catalogue.loc[catalogue.net_revenue_pkr_m >= 0]
rollup = positive.groupby(["category", "subcategory"], as_index=False).net_revenue_pkr_m.sum()
assert len(rollup) == 8
assert positive.shape[0] == 337
assert rollup.net_revenue_pkr_m.sum() == 1712.4508
print("Task 05 checks passed.")

Task 05 checks passed.
